In [ ]:
import sys, importlib, os

_src_training = os.path.dirname(os.path.abspath("training.ipynb"))
_src = os.path.dirname(_src_training)
for _p in [_src_training, _src]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import infrastructure, pipeline, classifier, evaluation
for _mod in [infrastructure, pipeline, classifier, evaluation]:
    importlib.reload(_mod)

from infrastructure import clean_neo4j_db, clean_kafka_topics, delete_test_neo4j_nodes, verify_concept_created
from pipeline import train_mnist, remove_concept, retrain_concept
from evaluation import test_mnist_all
from classifier import classify_image
import json, uuid
print("Modules loaded.")

In [ ]:
classes_to_subclasses = {
    0: [1],
    1: [1, 3],
    2: [1, 2],
    3: [1],
    4: [1, 2],
    5: [1],
    6: [1],
    7: [1],
    8: [1],
    9: [2],
}

In [ ]:
clean_neo4j_db()
clean_kafka_topics()

for class_num in classes_to_subclasses:
    for subclass in classes_to_subclasses[class_num]:
        train_mnist(class_number=class_num, subclass=subclass, is_prepared_samples=True, with_concept_creation=True)

In [ ]:
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    # "simplification_epsilon": 7,
}
results, y_true, y_pred, run_dir = test_mnist_all(list(classes_to_subclasses.keys()), params, sample_fraction=1.0)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

delete_test_neo4j_nodes()

class_number = 1
img_num = 448
image_id = f"mnist_{class_number}_{img_num:05d}"
local_path = f"../../tests/generated_samples/mnist_{class_number}/test"
nuclio_path = f"/opt/nuclio/shared_storage/generated_samples/mnist_{class_number}/test"

img_file = f"{image_id}.png"
img_local = os.path.join(local_path, img_file)
if os.path.exists(img_local):
    img = mpimg.imread(img_local)
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"MNIST Class {class_number}, Image #{img_num}")
    plt.axis("off")
    plt.show()

image_id_for_test = str(uuid.uuid4())
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "image_id": image_id_for_test,
    "session_id": "test",
    "delete_image_nodes": False,
    # "simplification_epsilon": 5,
}
result = classify_image(os.path.join(nuclio_path, img_file), params=params, timeout=60)
print(json.dumps(result, indent=2))

In [ ]:
# remove_concept("3_1")
# retrain_concept(number=3, subclass=1, with_concept_creation=False)